# 03 — Distribution model (schedule-level Monte Carlo)

**Runs only on a `GO` or `GO-TIER-B` verdict.** A win total is a *distribution* question — the OVER/UNDER price is
about P(wins > line), not about a point estimate. M4 (PREREGISTRATION §6) fits a prior-information
team rating, simulates every scheduled game of season *T*, and takes each team's win distribution
over simulations.

Simulating the actual schedule rather than sampling a parametric win count buys two things: league
wins are conserved by construction (every simulated game awards exactly one win), and opponent
quality enters through who a team actually plays.


> **STATUS: SCAFFOLD — no implementation.** The sections below are the planned structure, frozen for review before any code is written. Each will follow the repo's markdown → code → inline-test convention.

```bash
papermill futures/season_team_totals/03_distribution_model.ipynb /tmp/out.ipynb
```

## Parameters

In [ ]:
AUDIT_PATH   = None
PANEL_PATH   = None
N_SIMS       = 20000
SEED         = 20260802   # simulation is deterministic given this seed — tested, not assumed
TIE_PROB_MODEL = "empirical"   # NFL ties are rare but real; graded at half a win (§2.1)
WRITE_ARTIFACTS = True
RUN_TESTS    = True

## Planned sections

**Section 1 — Gate + load** — Audit gate, panel, frozen folds.

**Section 2 — Team ratings from prior information** — A rating per team-season fitted on training seasons only (prior record, point differential, EPA), plus a fitted home-field constant and a fitted residual scale.

**Section 3 — Game-level win probability** — Rating difference + home field → margin distribution → P(home win), P(tie). Tie probability from the empirical base rate, not assumed zero: ties settle at half a win and silently dropping them biases the distribution.

**Section 4 — Monte Carlo over the real schedule** — `N_SIMS` simulations of every scheduled game; per-team win counts accumulated with ties at 0.5. Seeded with `SEED`.

**Section 5 — Determinism and conservation tests** — Two runs at the same seed produce bit-identical win distributions; different seeds differ. In EVERY simulation, Σ team wins == number of games simulated. Per-team distribution support is within [0, games_played]. Probabilities sum to 1 per team.

**Section 6 — Push handling and settlement** — P(OVER), P(UNDER), P(PUSH) against the posted line. On an integer line P(PUSH) is real and must be non-zero; P(OVER)+P(UNDER)+P(PUSH)=1 exactly. On a half line P(PUSH)=0. The book's settlement assumptions (tie = half a win; cancelled games not made up) are stated in the output metadata, not left implicit.

**Section 7 — Distribution quality on the frozen folds** — CRPS vs the realized win count; log loss and Brier of P(OVER) with pushes excluded; 10-bin reliability; 50% and 80% central-interval coverage. Compared against the line-implied distribution wherever prices allow a de-vigged benchmark.

**Section 8 — Write** — `futures/artifacts/distribution_eval.json`. Research artifact; the live page does not read it.

## Not implemented

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SCAFFOLD — not implemented yet. Only `00_data_audit.ipynb` carries code today
# (Joseph reviews the audit before anything downstream is built), and the audit's
# current verdict is the gate this notebook would have to clear first.
#
# When implemented, this cell becomes the §5 gate: read futures/artifacts/data_audit.json,
# refuse to run on a NO-GO verdict, and read the FROZEN fold sets from it (headline + the A1.4 strict-subset sensitivity, which every reported number must carry) rather than
# choosing folds here.
# ─────────────────────────────────────────────────────────────────────────────
raise NotImplementedError(
    "futures/season_team_totals/03_distribution_model.ipynb is a scaffold. Sections are planned in the markdown cells above; "
    "implementation is gated on (1) review of 00_data_audit.ipynb and (2) a GO verdict "
    "in futures/artifacts/data_audit.json."
)

## Conclusion and next steps

**Status: scaffold — nothing implemented, nothing decided.** This notebook has produced no result and
written no artifact.

**Gate in force:** `futures/artifacts/data_audit.json` reads **`GO-TIER-B`** (2026-08-03) under
`PREREGISTRATION.md` §10 Amendment 1 — §7 gates **A and B** only, `tier_c_open: false`. The frozen
fold sets are the headline (10 test seasons) and the mandatory A1.4 strict-subset sensitivity
(4 test seasons, underpowered); both are read from the artifact, never recomputed here.

**On implementation this notebook must:** follow the repo's markdown → code → inline-test structure
with an explanation above and an interpretation below **every** code cell; report every headline
number twice (headline and A1.4 sensitivity); name the benchmark an *archived market consensus of
unattributed sportsbook origin*; and carry the §7 language fence — no sides, probabilities against a
posted line, confidence tiers, EV, or profitability, and none of the words *bet*, *edge*, *lock*,
*value*, *play*.

**Next step:** implement the sections planned above, in order, after the preceding notebook in the
pipeline has run and its artifact exists.